# Special Fixations

In [ ]:
import os

from main.frp_utils.path_helpers import *
from main.frp_utils.file_helpers import *
from main.frp_utils.aoi_helpers import *
from main.frp_utils.visual_helpers import *

In [ ]:
# get all files from participants
model = "deepseek-ai/deepseek-coder-1.3b-base"
short_model_name = model.split("/")[-1]
model_results_dir = f"../model_results/{short_model_name}"
code_snippets_with_data_driven_aois_dir = os.path.join(model_results_dir, "code_snippets_with_data_driven_aois")
AOI_SIZE_PATH = f'{code_snippets_with_data_driven_aois_dir}/aoi_sizes.csv'

folders = get_participant_folder_per_participant(raw_folder=False)
aoi_data = pd.read_csv(AOI_SIZE_PATH, sep=';', index_col=0)
print(folders)
display(aoi_data)

### Calculate special fixations with multiple algorithms

In [ ]:
participant_special_fixation_statistics = {}
participant_special_fixation_data = {}

for participant in tqdm(folders):
    print('----------------------------------------------')
    print(PARTICIPANT, participant)
    # skip participant if excluded
    print('(01/03) Load exclusions')
    exclusions = get_exclusions(participant, [PARTICIPANT], [VISUAL, BEHAVIORAL])[PARTICIPANT]
    if any(exclusions.values()):
        print('excluded')
        continue

    print('(02/03) Sort out previously excluded snippets')
    exclusions = get_exclusions(participant, [SNIPPET], [VISUAL, BEHAVIORAL])[SNIPPET]
    snippets = [snippet for snippet in exclusions if not any(exclusions[snippet].values())]
    print(len(snippets))

    print('(03/03) Identify special fixations')
    special_fixation_data, special_fixation_statistics = identify_special_fixations(participant, snippets, aoi_data)
    special_fixation_data.to_csv(get_special_fixations_path(participant), index=False, sep=';')
    participant_special_fixation_statistics[participant] = special_fixation_statistics
    participant_special_fixation_data[participant] = special_fixation_data

pd.concat(participant_special_fixation_statistics.values()).to_csv(get_special_fixation_statistics_path(), index=False, sep=';')
pd.concat(participant_special_fixation_data.values()).to_csv(get_special_fixations_path(), index=False, sep=';')